# Issue #9 — calibrated ranges and reconciled components

This notebook reviews the frozen issue #9 experiment. It reads the committed report and does not rerun final-test selection or scoring.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUTPUT = ROOT / 'benchmarks' / 'calibrated_intervals'
report = json.loads((OUTPUT / 'calibration_report.json').read_text(encoding='utf-8'))
comparison = pd.read_csv(OUTPUT / 'development_interval_comparison.csv')
report['selection']['policy']['method']

'horizon_asymmetric'

## What the earlier folds predicted

Each row was calibrated on earlier out-of-fold predictions and scored on a later fold. The final-test labels played no part in choosing the method.

In [2]:
comparison.groupby('method', as_index=False).agg(
    mean_forward_coverage=('coverage', 'mean'),
    mean_forward_width_mwh=('average_width_mwh', 'mean'),
    worst_forward_coverage=('coverage', 'min'),
).sort_values('mean_forward_coverage')

,method,mean_forward_coverage,mean_forward_width_mwh,worst_forward_coverage
2,horizon_recent_symmetric,0.727112,15.678353,0.552239
0,global_symmetric,0.795700,18.782803,0.558140
3,horizon_symmetric,0.796583,19.603409,0.609302
1,horizon_asymmetric,0.800513,21.614390,0.618605


## Sealed-period result

P50 improves, but interval coverage fails the 78–90% release target. The candidate stays experimental.

In [3]:
pd.DataFrame({
    'calibrated_candidate': report['final_test']['candidate'],
    'deployed_v1.1.0': report['final_test']['deployed_v1.1.0'],
}).loc[['p50_pinball_loss', 'coverage', 'average_width_mwh']]

,calibrated_candidate,deployed_v1.1.0
p50_pinball_loss,5.912513,6.690411
coverage,0.689095,0.772622
average_width_mwh,22.277215,34.109857


In [4]:
{'release_checks': report['release_checks'],
 'production_decision': report['production_decision'],
 'component_refit_eligible': report['components']['refit_eligibility']['eligible'],
 'component_reconciliation_error_mwh': report['components']['reconciliation_max_absolute_error_mwh']}

{'release_checks': {'components_reconciled': True,
  'coverage_max': True,
  'coverage_min': False,
  'p50_pinball': True,
  'quantiles_ordered_nonnegative': True,
  'width_not_excessive': True},
 'production_decision': 'retain_v1.1.0_intervals',
 'component_refit_eligible': False,
 'component_reconciliation_error_mwh': 2.842170943040401e-14}